# Phase 5 — Frozen Model A/B Test Generation

This notebook runs only after the Phase 5 freeze gate. It generates raw Test predictions on a Kaggle GPU and does not calculate metrics or select a model. Do not edit a frozen manifest, generation config, or input artifacts after opening Test.

In [ ]:
from pathlib import Path
import shutil
import subprocess

REPO = Path('/kaggle/working/VisolexNorm')
SOURCE_REF = 'refactor/review-weak-labels'
REPOSITORY_URL = 'https://github.com/AIVIETNAM-AIO-DinhBao/VisolexNorm.git'

if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', SOURCE_REF, REPOSITORY_URL, str(REPO)], check=True)
source_commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', source_commit], check=True)
print(f'Running refactored Phase 5 source at {source_commit}')

In [ ]:
%cd {REPO}
!pip install -q -r requirements-kaggle.txt

import platform, torch, transformers
print(platform.python_version(), torch.__version__, transformers.__version__)
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator before continuing.'

## Kaggle inputs

Attach datasets containing: (1) processed data with `data/processed/vilexnorm_test.jsonl` and `outputs/phase3_manifest.json`; (2) Model A checkpoint; (3) Model B checkpoint; (4) Phase 4 outputs containing `outputs/model_b/phase4_exit_report.json`; and (5) a **previously created**, immutable `freeze_manifest.json`. Update only the paths in the next cell before execution.

In [ ]:
# Kaggle mounts attached datasets by slug under /kaggle/input, not by owner URL.
DATA = Path('/kaggle/input/visolexnorm-processed')
MODEL_A = Path('/kaggle/input/phase2-output/checkpoints/model_a')
MODEL_B = Path('/kaggle/input/visolexnorm-phase3/checkpoints/model_b')
PHASE4_OUTPUTS = Path('/kaggle/input/visolexnorm-phase3/outputs/model_b')
FROZEN_MANIFEST = Path('/kaggle/input/visolexnorm-phase5-freeze/freeze_manifest.json')
WORK = Path('/kaggle/working/phase5')

TEST = DATA / 'data/processed/vilexnorm_test.jsonl'
PHASE3_MANIFEST = DATA / 'outputs/phase3_manifest.json'
PHASE4_EXIT_REPORT = PHASE4_OUTPUTS / 'phase4_exit_report.json'
CONFIG = REPO / 'configs/evaluation_generation_config.json'
METRIC_CODE = REPO / 'scripts/evaluation_metrics.py'
required_inputs = (MODEL_A, MODEL_B, TEST, PHASE3_MANIFEST, PHASE4_EXIT_REPORT, FROZEN_MANIFEST, CONFIG, METRIC_CODE)
missing = [str(path) for path in required_inputs if not path.exists()]
assert not missing, 'Missing required frozen input(s):\n' + '\n'.join(missing)
WORK.mkdir(parents=True, exist_ok=True)
print('Inputs ready:', *required_inputs, sep='\n- ')

In [ ]:
def run_checked(*command: str) -> None:
    print('$', ' '.join(command))
    subprocess.run(command, cwd=REPO, check=True)

run_checked('python', '-m', 'pytest', 'tests/contract/test_prediction_schema.py', 'tests/evaluation/test_frozen_metrics.py', 'tests/evaluation/test_freeze.py', 'tests/evaluation/test_predictions.py', '-q')

In [ ]:
run_checked('python', '-m', 'scripts.evaluation', 'verify-freeze', '--manifest', str(FROZEN_MANIFEST), '--model-a-checkpoint', str(MODEL_A), '--model-b-checkpoint', str(MODEL_B), '--test', str(TEST), '--generation-config', str(CONFIG), '--metric-code', str(METRIC_CODE), '--phase3-manifest', str(PHASE3_MANIFEST), '--phase4-exit-report', str(PHASE4_EXIT_REPORT))

## Generate Test predictions

The next two cells are the first allowed reads of Test. They fail closed when any frozen input differs from the manifest. Do not rerun training, change prompts/filters, or modify the generation config after this point.

In [ ]:
run_checked('python', '-m', 'scripts.evaluation', 'generate', '--manifest', str(FROZEN_MANIFEST), '--model', 'model_a', '--checkpoint', str(MODEL_A), '--model-a-checkpoint', str(MODEL_A), '--model-b-checkpoint', str(MODEL_B), '--test', str(TEST), '--generation-config', str(CONFIG), '--metric-code', str(METRIC_CODE), '--phase3-manifest', str(PHASE3_MANIFEST), '--phase4-exit-report', str(PHASE4_EXIT_REPORT), '--output', str(WORK / 'model_a_test_predictions.jsonl'))

In [ ]:
run_checked('python', '-m', 'scripts.evaluation', 'generate', '--manifest', str(FROZEN_MANIFEST), '--model', 'model_b', '--checkpoint', str(MODEL_B), '--model-a-checkpoint', str(MODEL_A), '--model-b-checkpoint', str(MODEL_B), '--test', str(TEST), '--generation-config', str(CONFIG), '--metric-code', str(METRIC_CODE), '--phase3-manifest', str(PHASE3_MANIFEST), '--phase4-exit-report', str(PHASE4_EXIT_REPORT), '--output', str(WORK / 'model_b_test_predictions.jsonl'))

In [ ]:
import json
import shutil
for model in ('model_a', 'model_b'):
    rows = [json.loads(line) for line in (WORK / f'{model}_test_predictions.jsonl').read_text(encoding='utf-8').splitlines()]
    assert len(rows) == 1045 and len({row['id'] for row in rows}) == 1045, f'Invalid output for {model}'
shutil.make_archive('/kaggle/working/phase5_test_predictions', 'zip', WORK)
print('Download phase5_test_predictions.zip. Do not calculate metrics in Kaggle; continue Phase 5 locally.')